# Predicting Run vs. Pass from Pre-Snap NFL Data
**CS 363M Machine Learning, Fall 2026**

## §0 Introduction
Before every snap, the defense has to guess: run or pass? We're building a model that makes that guess using only what's known **before the snap**: down, distance, field position, clock, score, weather, and who's lined up where.

**Data.** Two nflverse datasets from the 2022 season:
- **Play-by-play (pbp):** ~370 columns per play covering the game situation, clock, score, weather and outcome.
- **Participation:** formation, personnel, defenders in the box, plus some post-snap coverage/pressure data (from NFL Next Gen Stats).

**Task.** Binary classification on scrimmage plays: `is_pass = 1` for dropbacks, `0` for designed runs.

**Two scoping calls we made up front:**
- **Sacks and scrambles count as passes.** We're predicting the call, not the outcome, and those plays start as called passes. Calling them runs would just add noise.
- **Scrimmage plays only.** Our proposal said "pass vs. anything else," but punts and field goals are trivially predictable from 4th down plus formation. They'd pad our accuracy without the model learning anything. Kneels and spikes are clock management, so they're out too.

We explain each choice as we make it: what we picked, why, what else we considered, and where the data changed our plans.

**Contents**
1. Data acquisition and merge
2. Cleaning
3. Exploratory data analysis
4. Feature engineering
5. Train/test methodology
6. Modeling
7. Results and analysis

In [1]:
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 363

## §1 Data acquisition and merge

### 1.1 Download
We pull both parquet files from the `nflverse-data` GitHub releases and cache them in `data/` (gitignored). We went with parquet over CSV because it keeps the column types and loads much faster with this many columns. We also considered the `nflreadpy` package, but downloading directly means one less dependency.

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
BASE_URL = "https://github.com/nflverse/nflverse-data/releases/download"
FILES = {
    "pbp":  ("pbp", "play_by_play_2022.parquet"),
    "part": ("pbp_participation", "pbp_participation_2022.parquet"),
}

def fetch(release: str, fname: str) -> Path:
    """Download a file from an nflverse-data release unless it is already cached."""
    path = DATA_DIR / fname
    if not path.exists():
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(f"{BASE_URL}/{release}/{fname}", path)
    return path

pbp  = pd.read_parquet(fetch(*FILES["pbp"]))
part = pd.read_parquet(fetch(*FILES["part"]))
print(f"pbp:           {pbp.shape[0]:>6,} rows x {pbp.shape[1]} cols")
print(f"participation: {part.shape[0]:>6,} rows x {part.shape[1]} cols")

pbp:           49,434 rows x 372 cols
participation: 50,150 rows x 20 cols


In [3]:
part.head(3)

,nflverse_game_id,old_game_id,play_id,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,players_on_play,offense_players,defense_players,n_offense,n_defense,ngs_air_yards,time_to_throw,was_pressure,route,defense_man_zone_type,defense_coverage_type
0,2022_01_BUF_LA,2022090800,1,,None,None,NaN,None,NaN,,,,0,0,NaN,NaN,None,None,None,None
1,2022_01_BUF_LA,2022090800,41,LA,None,None,NaN,None,NaN,52512;42818;48482;52835;48772;54696;48011;5284...,00-0036421;00-0035933;00-0035609;00-0035942;00...,00-0031787;00-0035352;00-0037318;00-0035308;00...,11,11,NaN,NaN,None,None,None,None
2,2022_01_BUF_LA,2022090800,56,BUF,SHOTGUN,"1 RB, 1 TE, 3 WR",6.0,"3 DL, 3 LB, 5 DB",4.0,42816;48512;43298;47844;43335;47879;44875;4791...,00-0035382;00-0035689;00-0033904;00-0027648;00...,00-0031785;00-0033072;00-0035663;00-0032889;00...,11,11,4.52,2.169,False,IN,ZONE_COVERAGE,COVER_6


### 1.2 Join keys
The keys have different names in each table (`game_id` vs. `nflverse_game_id`), and pbp stores `play_id` as a float, so we line those up first. We also check that each table has exactly one row per play. Otherwise the join would quietly duplicate plays.

In [4]:
pbp["play_id"] = pbp["play_id"].astype("int64")
part["play_id"] = part["play_id"].astype("int64")
part = part.rename(columns={"nflverse_game_id": "game_id"})

KEYS = ["game_id", "play_id"]
assert not pbp.duplicated(KEYS).any(), "duplicate keys in pbp"
assert not part.duplicated(KEYS).any(), "duplicate keys in participation"
print("Games in pbp:", pbp.game_id.nunique(), "| games in participation:", part.game_id.nunique())
print("Season types in pbp:", pbp.season_type.value_counts().to_dict())

Games in pbp: 284 | games in participation: 284
Season types in pbp: {'REG': 47157, 'POST': 2277}


### 1.3 Join diagnostics
Before the real join, we do an **outer** join just to see what each side would lose.

In [5]:
diag = pbp[KEYS].merge(part[KEYS + ["offense_formation", "n_offense"]],
                       on=KEYS, how="outer", indicator=True)
print(diag["_merge"].value_counts().rename({"left_only": "pbp only",
                                            "right_only": "participation only",
                                            "both": "matched"}))

part_only = diag[diag["_merge"] == "right_only"]
print(f"\nParticipation-only rows with a formation recorded: {part_only.offense_formation.notna().sum()}")
print(f"Participation-only rows with any offensive players:  {(part_only.n_offense > 0).sum()}")

_merge
matched               49434
participation only      716
pbp only                  0
Name: count, dtype: int64

Participation-only rows with a formation recorded: 0
Participation-only rows with any offensive players:  0


**Takeaway.** Every pbp play has a match. The 716 participation-only rows are empty placeholders with no formation, no personnel and no players, so we lose nothing.

### 1.4 Inner join
**Why inner?** Our best features need both tables: situation from pbp and formation/personnel from participation. A left join would have meant making up formation data for unmatched plays. Since everything matched anyway, it's a free choice here.

We drop participation's `old_game_id` first, since pbp already has it.

In [6]:
df = pbp.merge(part.drop(columns=["old_game_id"]), on=KEYS, how="inner", validate="one_to_one")

assert len(df) == len(pbp), "inner join should keep every pbp play"
assert not df.duplicated(KEYS).any()
print(f"Merged: {df.shape[0]:,} rows x {df.shape[1]} cols")
df[["game_id", "play_id", "posteam", "down", "ydstogo", "play_type",
    "offense_formation", "offense_personnel", "defenders_in_box", "defense_personnel"]].head()

Merged: 49,434 rows x 389 cols


,game_id,play_id,posteam,down,ydstogo,play_type,offense_formation,offense_personnel,defenders_in_box,defense_personnel
0,2022_01_BAL_NYJ,1,None,NaN,0.0,None,None,None,NaN,None
1,2022_01_BAL_NYJ,43,NYJ,NaN,0.0,kickoff,None,None,NaN,None
2,2022_01_BAL_NYJ,68,NYJ,1.0,10.0,run,SINGLEBACK,"1 RB, 2 TE, 2 WR",7.0,"3 DL, 4 LB, 4 DB"
3,2022_01_BAL_NYJ,89,NYJ,1.0,10.0,pass,SHOTGUN,"1 RB, 2 TE, 2 WR",6.0,"3 DL, 4 LB, 4 DB"
4,2022_01_BAL_NYJ,115,NYJ,2.0,10.0,run,SINGLEBACK,"1 RB, 2 TE, 2 WR",7.0,"3 DL, 4 LB, 4 DB"


There's still a lot of junk in here: special teams, timeouts, and a ton of post-snap columns. That's what §2 is for.

## §2 Cleaning

Here we take the merged table, which still has every row nflverse records, and turn it into something we can model:
1. **Scope**: keep only scrimmage run/pass plays.
2. **Target**: define `is_pass` and check the class balance.
3. **Leakage audit**: whitelist the pre-snap columns, then double-check with data.
4. **Missing values**: handle each gap based on why it's missing.
5. **Categories**: check the long tails in formation and personnel.

Every filter logs how many rows it removes, so nothing disappears silently.

### 2.1 Scope: scrimmage run/pass plays
nflverse's `pass` and `rush` flags mark the offense's call, including sacks, scrambles and plays wiped out by a penalty after the snap. Rows with neither flag are special teams, kneels, spikes, timeouts and pre-snap penalties, so those go.

In [7]:
steps = [("merged", len(df))]

def log_step(name: str, frame: pd.DataFrame) -> pd.DataFrame:
    steps.append((name, len(frame)))
    return frame

print("play_type of rows with neither pass nor rush flag:")
print(df.loc[(df["pass"] != 1) & (df["rush"] != 1), "play_type"].value_counts(dropna=False).to_string())

clean = log_step("keep pass==1 or rush==1", df[(df["pass"] == 1) | (df["rush"] == 1)])
clean = log_step("drop 2-pt conversion tries", clean[clean["two_point_attempt"] != 1])
# 2-pt tries wiped out by a penalty are not flagged two_point_attempt, but they have no down
print("no-down plays left:", clean.loc[clean["down"].isna(), "desc"].str[:45].value_counts().to_dict())
clean = log_step("drop no-down plays (penalized 2-pt tries)", clean[clean["down"].notna()])
assert ((clean["pass"] + clean["rush"]) == 1).all(), 'pass and rush must be mutually exclusive'

play_type of rows with neither pass nor rush flag:
play_type
no_play        3123
kickoff        2835
punt           2294
None           1449
extra_point    1253
field_goal     1105
qb_kneel        442
qb_spike         63
no-down plays left: {'TWO-POINT CONVERSION ATTEMPT. 15-P.Mahomes pa': 2, 'TWO-POINT CONVERSION ATTEMPT. 3-R.Wilson pass': 2, 'TWO-POINT CONVERSION ATTEMPT. 4-D.Carr pass t': 1, 'TWO-POINT CONVERSION ATTEMPT. 16-J.Goff is sa': 1, 'TWO-POINT CONVERSION ATTEMPT. 10-J.Garoppolo ': 1, 'TWO-POINT CONVERSION ATTEMPT. 1-K.Murray pass': 1, 'TWO-POINT CONVERSION ATTEMPT. 12-T.Brady pass': 1, 'TWO-POINT CONVERSION ATTEMPT. 4-D.Ridder is s': 1, 'TWO-POINT CONVERSION ATTEMPT. 4-T.Heinicke pa': 1, 'TWO-POINT CONVERSION ATTEMPT. 17-J.Allen pass': 1, 'TWO-POINT CONVERSION ATTEMPT. 9-J.Burrow pass': 1}


**Aborted snaps.** When the snap gets fumbled, the called play never happens, and nflverse labels almost all of these `rush` no matter what was called:

In [8]:
aborted = clean[clean["aborted_play"] == 1]
print(f"{len(aborted)} aborted plays, labels: pass={int(aborted['pass'].sum())}, rush={int(aborted['rush'].sum())}")
print(aborted["desc"].head(3).str[:110].to_string())
clean = log_step("drop aborted snaps", clean[clean["aborted_play"] != 1])

113 aborted plays, labels: pass=1, rush=112
355    (7:36) (Shotgun) 6-B.Mayfield FUMBLES (Aborted...
463    (:39) (No Huddle, Shotgun) 6-B.Mayfield Aborte...
503    (1:27) 6-B.Mayfield FUMBLES (Aborted) at CLE 1...


So those labels are mostly noise. They're only 0.3% of plays, so we drop them.

Plays wiped out by a penalty (`no_play`) stay in as long as the flag tells us the call:

In [9]:
print(f"no_play rows kept: {(clean['play_type'] == 'no_play').sum():,}")
print(f"QB scrambles (play_type 'run', labelled pass): {((clean['play_type'] == 'run') & (clean['pass'] == 1)).sum():,}")

pd.DataFrame(steps, columns=["step", "rows"]).assign(removed=lambda t: -t["rows"].diff().fillna(0).astype(int))

no_play rows kept: 1,423
QB scrambles (play_type 'run', labelled pass): 959


,step,rows,removed
0,merged,49434,0
1,keep pass==1 or rush==1,36870,12564
2,drop 2-pt conversion tries,36742,128
3,drop no-down plays (penalized 2-pt tries),36729,13
4,drop aborted snaps,36616,113


### 2.2 Target
`is_pass` is just nflverse's `pass` flag. We considered counting only plays with an actual throw, but then every sack and scramble would be a "run" even though the offense called a pass.

In [10]:
clean = clean.copy()
clean["is_pass"] = clean["pass"].astype(int)

balance = clean["is_pass"].value_counts().rename({1: "pass", 0: "run"}).to_frame("plays")
balance["share"] = (balance["plays"] / len(clean)).round(3)
display(balance)
print(clean.groupby("season_type")["is_pass"].mean().round(3).rename("pass rate").to_string())
assert set(clean["is_pass"].unique()) == {0, 1}

,plays,share
is_pass,,
pass,22353,0.61
run,14263,0.39


season_type
POST    0.639
REG     0.609


**Takeaway.** 61% of plays are passes (64% in the playoffs), so "always pass" already gets 61% accuracy. That's why we use ROC-AUC as our main metric: accuracy flatters a model that just leans "pass." We'll also try class weighting in §6.

### 2.3 Leakage audit
Most of the ~390 columns describe what happened after the snap. We could list the leaky ones and drop them, but missing even one (say, `qb_dropback`) hands the model the answer. So we flip it around: we **whitelist** the pre-snap columns and drop everything else by default.

The columns we keep play four roles:
- **Feature candidates**: known before the snap; §4 builds features from these.
- **IDs**: for joining, splitting and grouping. Never fed to a model.
- **Auxiliary**: `yards_gained` is post-snap, but we need it to build "how many yards did the previous play gain?" in §4. An assert makes sure it never ends up in X.
- **Benchmark**: `xpass` is nflverse's own pass-probability model. Using it as a feature would be copying their homework, so we only compare against it in §7. (Its sibling `pass_oe` literally contains the label.)

In [11]:
ID_COLS = ["game_id", "play_id", "week", "season_type", "posteam", "defteam",
           "home_team", "fixed_drive"]
AUX_COLS = ["yards_gained"]   # post-snap: only for lagged features of the previous play
BENCHMARK_COLS = ["xpass"]    # never a feature: nflverse's own model, kept only as a benchmark
TARGET = "is_pass"

FEATURE_CANDIDATES = [
    # game situation (pbp)
    "down", "ydstogo", "yardline_100", "goal_to_go", "qtr",
    "quarter_seconds_remaining", "half_seconds_remaining", "game_seconds_remaining",
    "score_differential", "posteam_timeouts_remaining", "defteam_timeouts_remaining",
    "posteam_type", "wp", "spread_line", "total_line",
    # pre-snap alignment (pbp)
    "shotgun", "no_huddle",
    # personnel and formation (participation)
    "offense_formation", "offense_personnel", "defense_personnel", "defenders_in_box",
    # venue and weather (pbp)
    "roof", "surface", "temp", "wind", "weather",
]
KEEP = ID_COLS + AUX_COLS + BENCHMARK_COLS + FEATURE_CANDIDATES + [TARGET]
assert len(KEEP) == len(set(KEEP))
excluded = [c for c in clean.columns if c not in KEEP]
print(f"Keeping {len(KEEP)} of {clean.shape[1]} columns; excluding {len(excluded)}.")

Keeping 37 of 390 columns; excluding 353.


Here's why everything else got cut, grouped by rule:

In [12]:
import re

EXCLUSION_RULES = [
    ("label or label-derived (pass_oe = pass - xpass)",
     r"^(pass|rush|pass_oe|pass_attempt|rush_attempt|qb_dropback|qb_scramble|sack|play_type|play_type_nfl|desc|aborted_play|two_point_attempt)$"),
    ("post-snap outcome (yards, EPA/WPA, scores after the play, result flags)",
     r"yards|epa|wpa|_post$|touchdown|first_down|complete|interception|fumble|penalty|safety|success|"
     r"air_|yac|return|tackle|pass_length|pass_location|run_location|run_gap|out_of_bounds|lateral|"
     r"^td_|_td$|conversion|^(ep|wp)a?$|^cp|^cpoe|^xyac|^comp_|^total_|^posteam_score$|^defteam_score$|"
     r"replay|qb_hit|qb_kneel|qb_spike|assist|forced|blocked|downed|touchback|fair_catch|own_kickoff|kick|punt|field_goal|extra_point|timeout$"),
    ("player identity (who touched the ball reveals the play type)",
     r"_player|passer|rusher|receiver|^fantasy|jersey|^name$|^id$|^players_on_play$|_players$"),
    ("drive/series summary (includes plays that have not happened yet)",
     r"^drive_|fixed_drive_result|series"),
    ("participation post-snap (NGS)",
     r"number_of_pass_rushers|time_to_throw|was_pressure|^route$|ngs_air_yards|coverage|man_zone|^n_offense$|^n_defense$"),
]

def reason_for(col: str) -> str:
    for reason, pattern in EXCLUSION_RULES:
        if re.search(pattern, col):
            return reason
    return "not needed (identifier, metadata, redundant, or special teams)"

audit = pd.DataFrame({"column": excluded, "reason": [reason_for(c) for c in excluded]})
(audit.groupby("reason")["column"]
      .agg(n="count", examples=lambda s: ", ".join(s.head(8)))
      .sort_values("n", ascending=False))

,n,examples
reason,,
"post-snap outcome (yards, EPA/WPA, scores after the play, result flags)",224,"qb_kneel, qb_spike, pass_length, pass_location..."
"not needed (identifier, metadata, redundant, or special teams)",55,"old_game_id, away_team, side_of_field, game_da..."
player identity (who touched the ball reveals the play type),36,"passer_player_id, passer_player_name, receiver..."
drive/series summary (includes plays that have not happened yet),18,"series, series_result, fixed_drive_result, dri..."
label or label-derived (pass_oe = pass - xpass),13,"desc, play_type, qb_dropback, qb_scramble, rus..."
participation post-snap (NGS),7,"n_offense, n_defense, time_to_throw, was_press..."


**Double-checking with data.** The whitelist only catches leaks we thought of. To catch the rest, we score every numeric column on how well it separates passes from runs by itself (univariate ROC-AUC, where 0.5 = no signal and 1.0 = perfect). A leaky column will score near 1.0.

We score each column twice: once on its **values**, and once on whether it's **missing**. Some columns, like `air_yards`, only exist on passes, so just being filled in gives the answer away.

In [13]:
from sklearn.metrics import roc_auc_score

def univariate_auc(frame: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    rows = []
    for col in frame.select_dtypes(include="number").columns:
        x = frame[col]
        scores = {}
        ok = x.notna()
        if ok.sum() > 100 and x[ok].nunique() > 1 and y[ok].nunique() > 1:
            scores["value"] = roc_auc_score(y[ok], x[ok])
        if 0 < ok.mean() < 1:
            scores["missingness"] = roc_auc_score(y, ok.astype(int))
        if scores:
            kind, auc = max(scores.items(), key=lambda kv: abs(kv[1] - 0.5))
            rows.append((col, kind, max(auc, 1 - auc)))
    return pd.DataFrame(rows, columns=["column", "via", "auc"]).sort_values("auc", ascending=False)

leak_scan = univariate_auc(clean.drop(columns=[TARGET]), clean[TARGET])
leak_scan["status"] = np.where(leak_scan["column"].isin(FEATURE_CANDIDATES), "FEATURE", "excluded")
leak_scan.head(15)

,column,via,auc,status
194,pass_oe,value,1.000000,excluded
181,pass,value,1.000000,excluded
182,rush,value,1.000000,excluded
178,passer_jersey_number,missingness,0.999978,excluded
179,rusher_jersey_number,missingness,0.976158,excluded
16,qb_dropback,value,0.975484,excluded
196,number_of_pass_rushers,missingness,0.971012,excluded
119,rush_attempt,value,0.967257,excluded
140,rushing_yards,missingness,0.967086,excluded
120,pass_attempt,value,0.954745,excluded


In [14]:
top_features = leak_scan[leak_scan["status"] == "FEATURE"].head(8)
display(top_features)

LEAK_THRESHOLD = 0.90
suspicious = leak_scan[(leak_scan["auc"] > LEAK_THRESHOLD) & (leak_scan["status"] == "FEATURE")]
assert suspicious.empty, f"possible leakage in whitelisted features:\n{suspicious}"
for c in ["xpass", "pass_oe", "yards_gained", "air_yards", "qb_dropback", "number_of_pass_rushers"]:
    assert c not in FEATURE_CANDIDATES
print(f"No whitelisted feature exceeds univariate AUC {LEAK_THRESHOLD}.")
print(f"Benchmark xpass univariate AUC: {leak_scan.set_index('column').loc['xpass', 'auc']:.3f}")

,column,via,auc,status
195,defenders_in_box,value,0.714378,FEATURE
14,shotgun,value,0.688564,FEATURE
9,down,value,0.619675,FEATURE
60,wp,value,0.608434,FEATURE
29,score_differential,value,0.583561,FEATURE
4,half_seconds_remaining,value,0.552248,FEATURE
3,quarter_seconds_remaining,value,0.548490,FEATURE
11,ydstogo,value,0.547369,FEATURE


No whitelisted feature exceeds univariate AUC 0.9.
Benchmark xpass univariate AUC: 0.788


**Takeaway.** Everything near 1.0 is post-snap or label-derived, and it's all excluded. Note that several leak only through missingness (`passer_jersey_number` only exists if someone passed), which is exactly what a hand-made drop list would miss. Our best pre-snap features are `defenders_in_box` (0.71) and `shotgun` (0.69): useful, sensible, and nowhere near perfect, which is what real signal looks like. `xpass` scores 0.79 because it's a whole model, which makes it a good benchmark.

Now we cut down to the whitelist:

In [15]:
clean = clean[KEEP].reset_index(drop=True)
print(clean.shape)

(36616, 37)


### 2.4 Missing values

In [16]:
miss = clean[FEATURE_CANDIDATES].isna().sum()
miss = miss[miss > 0].to_frame("missing")
miss["pct"] = (100 * miss["missing"] / len(clean)).round(2)
miss

,missing,pct
offense_formation,76,0.21
offense_personnel,1,0.00
defense_personnel,1,0.00
defenders_in_box,74,0.20
temp,22878,62.48
wind,22878,62.48
weather,1196,3.27


Three different reasons for missing data, so three different fixes:

| Column(s) | Why it's missing | What we do |
|---|---|---|
| formation, personnel, `defenders_in_box` | Tracking gap on a handful of plays | **Drop the rows**: under 0.3%, and imputing would mean inventing formation data |
| `temp`, `wind` (dome/closed roof) | No weather indoors | `is_indoor = 1`, set to 70°F and 0 mph |
| `temp`, `wind` (outdoor/open roof) | nflverse just didn't fill them in, **but the `weather` text has the numbers** | **Parse them out of the text** |

**This changed our plan.** We originally assumed missing `temp` always meant indoors and were going to fill in 70°F and no wind everywhere. Turns out ~10,000 outdoor plays from 81 games are missing it too:

In [17]:
print(pd.crosstab(clean["roof"], clean["temp"].isna().rename("temp missing")))
print("\nSample weather strings:", clean["weather"].dropna().drop_duplicates().head(3).tolist(), sep="\n")

temp missing  False  True 
roof                      
closed            0   4262
dome              0   6865
open              0   1313
outdoors      13738  10438

Sample weather strings:
['Rain and mid 70s Temp: 73° F, Humidity: 79%, Wind: W 3 mph', 'Cloudy Temp: 88° F, Humidity: 48%, Wind: W 8 mph', 'Cloudy Temp: 80° F, Humidity: 81%, Wind: NE 8 mph']


In [18]:
# 1) NGS tracking gaps -> drop rows
ngs_cols = ["offense_formation", "offense_personnel", "defense_personnel", "defenders_in_box"]
clean = log_step("drop missing formation/personnel/box", clean.dropna(subset=ngs_cols))

# 2) indoor games: weather does not apply
clean["is_indoor"] = clean["roof"].isin(["dome", "closed"]).astype(int)
clean.loc[clean["is_indoor"] == 1, ["temp", "wind"]] = [70.0, 0.0]

# 3) outdoor games with missing numbers: parse the weather text
weather_txt = clean["weather"].fillna("")
parsed_temp = pd.to_numeric(weather_txt.str.extract(r"Temp:\s*(-?\d+)", expand=False))
parsed_wind = pd.to_numeric(weather_txt.str.extract(r"Wind:\s*[A-Za-z]*\s*(\d+)\s*mph", expand=False))
need_temp, need_wind = clean["temp"].isna(), clean["wind"].isna()
clean.loc[need_temp, "temp"] = parsed_temp[need_temp]
clean.loc[need_wind, "wind"] = parsed_wind[need_wind]
print(f"temp filled from weather text: {(need_temp & clean['temp'].notna()).sum():,}")
print(f"wind filled from weather text: {(need_wind & clean['wind'].notna()).sum():,}")

# 4) small text clean-ups
clean["surface"] = clean["surface"].replace("", "unknown")
clean["weather"] = clean["weather"].fillna("")   # parsed into rain/snow flags in §4

remaining = clean[FEATURE_CANDIDATES].isna().sum()
print("\nStill missing:", remaining[remaining > 0].to_dict() or "none")

temp filled from weather text: 11,721
wind filled from weather text: 11,721

Still missing: none


Parsing got every value back, so there's nothing left to impute. We'll still put a median imputer inside the Pipeline just in case. That way, if new data has gaps, the fill value comes from training folds only. The parsed values look reasonable (6–92°F, 0–27 mph):

In [19]:
# sanity check on the parsed values
clean[["temp", "wind"]].describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
temp,36539.0,60.3,16.5,6.0,52.0,68.0,70.0,92.0
wind,36539.0,5.9,5.6,0.0,0.0,5.0,10.0,27.0


### 2.5 Categorical cardinality

In [20]:
form = clean["offense_formation"].value_counts().to_frame("plays")
form["share_%"] = (100 * form["plays"] / len(clean)).round(2)
display(form)

for col in ["offense_personnel", "defense_personnel"]:
    vc = clean[col].value_counts(normalize=True)
    print(f"{col}: {len(vc)} distinct strings; top 5 cover {vc.head(5).sum():.1%}, "
          f"{(vc < 0.005).sum()} groupings are under 0.5% of plays each")
print("\nexample odd strings:", clean.loc[clean["offense_personnel"].str.contains("DB|DL|LB|QB"),
                                         "offense_personnel"].unique()[:4])

,plays,share_%
offense_formation,,
SHOTGUN,20300,55.56
SINGLEBACK,8859,24.25
EMPTY,3003,8.22
I_FORM,2227,6.09
PISTOL,1578,4.32
JUMBO,312,0.85
WILDCAT,260,0.71


offense_personnel: 64 distinct strings; top 5 cover 94.8%, 56 groupings are under 0.5% of plays each
defense_personnel: 64 distinct strings; top 5 cover 82.8%, 51 groupings are under 0.5% of plays each

example odd strings: ['1 RB, 1 TE, 2 WR,1 LB' '2 QB, 1 RB, 1 TE, 2 WR' '2 QB, 1 RB, 2 TE, 1 WR'
 '2 QB, 1 RB, 0 TE, 3 WR']


**Takeaway.**
- **Formation:** all 7 formations show up on at least 0.5% of plays (WILDCAT is rarest at 0.7%), so nothing needs merging.
- **Personnel:** 64 distinct groupings on each side, with a long tail. The top 5 cover 95% of offensive and 83% of defensive snaps, while 50+ groupings each show up on under 0.5% of plays, including weird ones like a linebacker on offense or two QBs. One-hot encoding all of them would make dozens of nearly empty columns. In §4 we'll:
  1. Parse each grouping into **counts** (`n_rb`, `n_te`, `n_wr`, …), so rare groupings still look like their common cousins.
  2. Let `OneHotEncoder(min_frequency=0.005)` lump rare categories together inside the Pipeline.

  We originally planned to merge rare groupings into "OTHER" up front. We moved that into the Pipeline because picking the cut-off from the full dataset would sneak in test-set info.

### 2.6 Cleaning summary

In [21]:
summary = pd.DataFrame(steps, columns=["step", "rows"])
summary["removed"] = -summary["rows"].diff().fillna(0).astype(int)
display(summary)

assert not clean.duplicated(["game_id", "play_id"]).any()
assert clean[TARGET].isin([0, 1]).all()
assert "yards_gained" not in FEATURE_CANDIDATES and "xpass" not in FEATURE_CANDIDATES
print(f"Clean table: {clean.shape[0]:,} plays x {clean.shape[1]} columns | pass rate {clean[TARGET].mean():.3f}")

,step,rows,removed
0,merged,49434,0
1,keep pass==1 or rush==1,36870,12564
2,drop 2-pt conversion tries,36742,128
3,drop no-down plays (penalized 2-pt tries),36729,13
4,drop aborted snaps,36616,113
5,drop missing formation/personnel/box,36539,77


Clean table: 36,539 plays x 38 columns | pass rate 0.610
